# K-Means Clustering

## 1. Introduction

### What is Clustering?

Clustering is the task of grouping a set of objects so that objects in the same group (called a **cluster**) are more similar to each other than to those in other groups. Unlike classification, clustering is an **unsupervised learning** technique — we do not have labeled target values. The algorithm must discover the structure in the data on its own.

### Unsupervised Learning

In **supervised learning** (e.g., linear regression, logistic regression, decision trees), we train a model on input-output pairs. In **unsupervised learning**, we only have input data and the goal is to find hidden patterns or intrinsic structures.

### What is K-Means?

**K-Means** is one of the simplest and most popular unsupervised clustering algorithms. Given a dataset and a number **k**, K-Means partitions the data into exactly **k** clusters by minimizing the within-cluster sum of squares (the total squared distance between each point and its assigned cluster centroid).

Key properties:
- Simple and fast
- Works well when clusters are roughly spherical and equally sized
- Requires the user to specify **k** in advance
- Guaranteed to converge, but may converge to a local optimum

## 2. Algorithm Overview

The K-Means algorithm repeats three steps until convergence:

1. **Initialize Centroids** — Pick k initial centroid positions (e.g., randomly select k data points).

2. **Assign Clusters** — For each data point, compute the distance to every centroid and assign the point to the cluster of the nearest centroid.

3. **Update Centroids** — For each cluster, compute the mean of all points assigned to it. This mean becomes the new centroid.

4. **Repeat** steps 2–3 until the centroids no longer change (or change by less than a small threshold ε).

```
Initialize k centroids
Repeat:
    For each point:
        Assign to nearest centroid
    For each cluster:
        Recompute centroid as mean of assigned points
Until centroids converge
```

## 3. Distance Function

We need a way to measure how "close" two points are. The most common choice is **Euclidean distance**:

$$d(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

where $\mathbf{a}$ and $\mathbf{b}$ are n-dimensional vectors.

In [ ]:
from math import sqrt


def euclidean_distance(point_a, point_b):
    """Compute the Euclidean distance between two points (lists of equal length)."""
    return sqrt(sum((a - b) ** 2 for a, b in zip(point_a, point_b)))


# Quick test
print(euclidean_distance([0, 0], [3, 4]))  # Expected: 5.0
print(euclidean_distance([1, 2, 3], [4, 6, 3]))  # Expected: 5.0

## 4. Centroid Initialization

The simplest initialization strategy is to randomly select **k** distinct data points as the initial centroids. The quality of the final clustering can depend heavily on this initial choice — we'll discuss this more in the Limitations section.

In [ ]:
from random import seed, sample


def initialize_centroids(dataset, k):
    """Randomly select k data points as initial centroids."""
    return [list(point) for point in sample(dataset, k)]


# Quick test
seed(42)
test_data = [[1, 2], [3, 4], [5, 6], [7, 8], [9, 10]]
centroids = initialize_centroids(test_data, 2)
print('Initial centroids:', centroids)

## 5. Cluster Assignment

Given a set of centroids, we assign each data point to the cluster whose centroid is closest. The function returns a list of cluster indices (0, 1, ..., k-1), one per data point.

In [ ]:
def assign_clusters(dataset, centroids):
    """Assign each point to the index of the nearest centroid."""
    assignments = []
    for point in dataset:
        distances = [euclidean_distance(point, c) for c in centroids]
        nearest = distances.index(min(distances))
        assignments.append(nearest)
    return assignments


# Quick test
test_centroids = [[1, 1], [9, 9]]
test_points = [[0, 0], [2, 2], [8, 8], [10, 10]]
print(assign_clusters(test_points, test_centroids))  # Expected: [0, 0, 1, 1]

## 6. Centroid Update

After assigning points to clusters, we recompute each centroid as the **mean** (component-wise average) of all points currently in that cluster.

$$\mathbf{c}_j = \frac{1}{|S_j|} \sum_{\mathbf{x} \in S_j} \mathbf{x}$$

where $S_j$ is the set of points assigned to cluster $j$.

In [ ]:
def update_centroids(dataset, assignments, k):
    """Compute new centroids as the mean of points in each cluster."""
    n_features = len(dataset[0])
    new_centroids = []
    for cluster_idx in range(k):
        cluster_points = [dataset[i] for i in range(len(dataset)) if assignments[i] == cluster_idx]
        if len(cluster_points) == 0:
            # If a cluster lost all points, reinitialize to a random data point
            new_centroids.append(list(sample(dataset, 1)[0]))
        else:
            centroid = []
            for f in range(n_features):
                centroid.append(sum(p[f] for p in cluster_points) / len(cluster_points))
            new_centroids.append(centroid)
    return new_centroids


# Quick test
test_data = [[0, 0], [2, 2], [8, 8], [10, 10]]
test_assignments = [0, 0, 1, 1]
print(update_centroids(test_data, test_assignments, 2))  # Expected: [[1.0, 1.0], [9.0, 9.0]]

## 7. Convergence Check

The algorithm converges when the centroids stop moving. In practice, we check whether the total shift of all centroids is below a small threshold **ε** (epsilon), or simply whether the old and new centroids are identical.

In [ ]:
def has_converged(old_centroids, new_centroids, epsilon=1e-6):
    """Check if centroids have stopped moving (total shift < epsilon)."""
    total_shift = sum(euclidean_distance(old, new) for old, new in zip(old_centroids, new_centroids))
    return total_shift < epsilon


# Quick test
print(has_converged([[1, 1], [5, 5]], [[1, 1], [5, 5]]))          # True — no movement
print(has_converged([[1, 1], [5, 5]], [[1.1, 1.1], [5.1, 5.1]]))  # False — still moving

## 8. Full K-Means Implementation

Now we combine all the pieces into a single `k_means` function.

In [ ]:
def k_means(dataset, k, max_iterations=100, epsilon=1e-6):
    """
    Run K-Means clustering on the dataset.

    Parameters:
        dataset        : list of data points (each point is a list of floats)
        k              : number of clusters
        max_iterations : maximum number of iterations before stopping
        epsilon        : convergence threshold for centroid movement

    Returns:
        centroids   : final centroid positions (list of k points)
        assignments : cluster index for each data point
    """
    centroids = initialize_centroids(dataset, k)

    for iteration in range(max_iterations):
        assignments = assign_clusters(dataset, centroids)
        new_centroids = update_centroids(dataset, assignments, k)

        if has_converged(centroids, new_centroids, epsilon):
            print(f'Converged after {iteration + 1} iteration(s).')
            return new_centroids, assignments

        centroids = new_centroids

    print(f'Stopped after reaching max iterations ({max_iterations}).')
    return centroids, assignments

## 9. Worked Example

Let's create a small 2D dataset with **3 obvious clusters** and run our K-Means algorithm on it.

In [ ]:
from random import uniform, seed as rseed


def generate_cluster(center_x, center_y, spread, n):
    """Generate n 2D points around a center with given spread."""
    return [[center_x + uniform(-spread, spread),
             center_y + uniform(-spread, spread)] for _ in range(n)]


rseed(7)

# Three well-separated clusters
cluster_a = generate_cluster(2, 2, 1.0, 20)    # Bottom-left
cluster_b = generate_cluster(8, 3, 1.0, 20)    # Bottom-right
cluster_c = generate_cluster(5, 9, 1.0, 20)    # Top-center

dataset = cluster_a + cluster_b + cluster_c

print(f'Dataset size: {len(dataset)} points')
print(f'First 5 points: {dataset[:5]}')

In [ ]:
# Run K-Means with k=3
rseed(42)
centroids, assignments = k_means(dataset, k=3)

print('\nFinal centroids:')
for i, c in enumerate(centroids):
    print(f'  Cluster {i}: ({c[0]:.2f}, {c[1]:.2f})')

# Count points per cluster
for i in range(3):
    count = assignments.count(i)
    print(f'  Cluster {i}: {count} points')

In [ ]:
# Visualize the clustering result (using matplotlib for visualization only — the algorithm is pure Python)
try:
    import matplotlib.pyplot as plt

    colors = ['#e74c3c', '#3498db', '#2ecc71']
    markers = ['o', 's', '^']

    plt.figure(figsize=(8, 6))

    for i in range(3):
        cluster_points = [dataset[j] for j in range(len(dataset)) if assignments[j] == i]
        xs = [p[0] for p in cluster_points]
        ys = [p[1] for p in cluster_points]
        plt.scatter(xs, ys, c=colors[i], marker=markers[i], label=f'Cluster {i}', alpha=0.7, s=60)

    cx = [c[0] for c in centroids]
    cy = [c[1] for c in centroids]
    plt.scatter(cx, cy, c='black', marker='X', s=200, zorder=5, label='Centroids')

    plt.title('K-Means Clustering (k=3)')
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

except ImportError:
    print('matplotlib not available — skipping visualization.')

## 10. Elbow Method — Choosing k

A common question is: **how do we choose the right value of k?**

The **Elbow Method** is a heuristic that helps:

1. Run K-Means for k = 1, 2, 3, ..., K.
2. For each k, compute the **Within-Cluster Sum of Squares (WCSS)**: $$\text{WCSS} = \sum_{j=1}^{k} \sum_{\mathbf{x} \in S_j} \|\mathbf{x} - \mathbf{c}_j\|^2$$
3. Plot WCSS vs. k. The "elbow" — the point where adding more clusters yields diminishing returns — suggests a good value of k.

In [ ]:
def compute_wcss(dataset, centroids, assignments):
    """Compute the Within-Cluster Sum of Squares."""
    wcss = 0.0
    for i, point in enumerate(dataset):
        centroid = centroids[assignments[i]]
        wcss += sum((a - b) ** 2 for a, b in zip(point, centroid))
    return wcss


# Run K-Means for k = 1 to 8 and record WCSS
wcss_values = []
k_range = range(1, 9)

for k in k_range:
    rseed(42)
    c, a = k_means(dataset, k)
    w = compute_wcss(dataset, c, a)
    wcss_values.append(w)
    print(f'k={k}  WCSS={w:.2f}')

In [ ]:
# Plot the Elbow curve
try:
    import matplotlib.pyplot as plt

    plt.figure(figsize=(8, 5))
    plt.plot(list(k_range), wcss_values, 'bo-', linewidth=2, markersize=8)
    plt.title('Elbow Method — WCSS vs. k')
    plt.xlabel('Number of clusters (k)')
    plt.ylabel('Within-Cluster Sum of Squares (WCSS)')
    plt.xticks(list(k_range))
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print('Look for the "elbow" — the k after which WCSS decreases much more slowly.')
    print('For our dataset with 3 true clusters, the elbow should appear at k=3.')

except ImportError:
    print('matplotlib not available — skipping plot.')
    print('WCSS values:', list(zip(k_range, wcss_values)))

## 11. Limitations of K-Means

### 1. Sensitivity to Initialization

Because K-Means uses random initialization, different runs can produce different clusterings. A poor initial choice can lead the algorithm to converge to a **local minimum** rather than the global optimum.

A common mitigation is to run K-Means multiple times with different random seeds and keep the result with the lowest WCSS.

### 2. K-Means++ Initialization

**K-Means++** is a smarter initialization strategy (Arthur & Vassilvitskii, 2007):

1. Choose the first centroid uniformly at random from the data.
2. For each subsequent centroid, choose a data point with probability proportional to its squared distance from the nearest existing centroid.
3. Repeat until k centroids are chosen.

This spreads out the initial centroids and almost always leads to better results. Let's implement it:

In [ ]:
from random import random, choice


def initialize_centroids_pp(dataset, k):
    """K-Means++ initialization: select centroids spread apart."""
    centroids = [list(choice(dataset))]

    for _ in range(1, k):
        # Squared distance from each point to its nearest existing centroid
        distances_sq = []
        for point in dataset:
            min_dist_sq = min(sum((a - b) ** 2 for a, b in zip(point, c)) for c in centroids)
            distances_sq.append(min_dist_sq)

        # Weighted random selection (roulette wheel)
        total = sum(distances_sq)
        r = random() * total
        cumulative = 0.0
        for i, d_sq in enumerate(distances_sq):
            cumulative += d_sq
            if cumulative >= r:
                centroids.append(list(dataset[i]))
                break

    return centroids


# Test K-Means++ initialization
rseed(42)
pp_centroids = initialize_centroids_pp(dataset, 3)
print('K-Means++ initial centroids:')
for i, c in enumerate(pp_centroids):
    print(f'  Centroid {i}: ({c[0]:.2f}, {c[1]:.2f})')

In [ ]:
def k_means_pp(dataset, k, max_iterations=100, epsilon=1e-6):
    """K-Means with K-Means++ initialization."""
    centroids = initialize_centroids_pp(dataset, k)

    for iteration in range(max_iterations):
        assignments = assign_clusters(dataset, centroids)
        new_centroids = update_centroids(dataset, assignments, k)

        if has_converged(centroids, new_centroids, epsilon):
            print(f'K-Means++ converged after {iteration + 1} iteration(s).')
            return new_centroids, assignments

        centroids = new_centroids

    print(f'K-Means++ stopped after max iterations ({max_iterations}).')
    return centroids, assignments


rseed(42)
centroids_pp, assignments_pp = k_means_pp(dataset, k=3)

print('\nK-Means++ final centroids:')
for i, c in enumerate(centroids_pp):
    print(f'  Cluster {i}: ({c[0]:.2f}, {c[1]:.2f})')

for i in range(3):
    print(f'  Cluster {i}: {assignments_pp.count(i)} points')

### 3. Assumes Spherical (Globular) Clusters

K-Means uses Euclidean distance and assigns points to the nearest centroid. This naturally creates **Voronoi regions** that carve space into convex polygons. As a result, K-Means performs poorly on:
- Elongated clusters
- Ring-shaped / concentric clusters
- Clusters of very different sizes or densities

For such cases, algorithms like **DBSCAN** or **Gaussian Mixture Models** may be more appropriate.

### 4. k Must Be Specified in Advance

Unlike some clustering methods (e.g., DBSCAN, hierarchical clustering), K-Means requires the user to decide the number of clusters beforehand. The **Elbow Method** (Section 10) and the **Silhouette Score** are two common approaches for choosing k.

### 5. Sensitive to Outliers

Because the centroid is the **mean** of all cluster members, a single outlier far from the true cluster center can shift the centroid significantly. Variants like **K-Medoids** (which use the actual data point closest to the center) are more robust to outliers.

## 12. Summary

In this notebook we implemented K-Means clustering from scratch in pure Python:

| Component | Function | Purpose |
|---|---|---|
| Distance | `euclidean_distance` | Measure proximity between points |
| Initialization | `initialize_centroids` | Random selection of k starting centroids |
| Assignment | `assign_clusters` | Assign each point to the nearest centroid |
| Update | `update_centroids` | Recompute centroids as cluster means |
| Convergence | `has_converged` | Stop when centroids stabilize |
| Full algorithm | `k_means` | Combine all steps into an iterative procedure |
| WCSS | `compute_wcss` | Evaluate clustering quality |
| K-Means++ | `initialize_centroids_pp` / `k_means_pp` | Smarter initialization for better results |

### Key Takeaways

- K-Means is a simple, efficient clustering algorithm that partitions data into **k** groups.
- The algorithm alternates between **assigning points** and **updating centroids** until convergence.
- The **Elbow Method** helps choose a good value of k by looking at the WCSS curve.
- **K-Means++ initialization** significantly improves reliability over naive random initialization.
- K-Means assumes spherical clusters, is sensitive to outliers, and requires k to be specified.

### What's Next?

With K-Means as a foundation, you can explore:
- **K-Medoids** — more robust to outliers
- **DBSCAN** — density-based clustering, no need to specify k
- **Hierarchical Clustering** — builds a tree of clusters
- **Gaussian Mixture Models** — soft (probabilistic) cluster assignments